In [ ]:
#load in modules
import os
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import scipy.stats as stats 
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import time
from tabulate import tabulate
import xarray as xr #to join to dataset
import rasterio #for handling raster data

Definining the data completeness function 

In [7]:
def summarize_dataframe(df):
    """
    Prints a summary of the DataFrame's columns including:
    - dtype
    - count of non-null values
    - % completeness
    - number of unique values
    Output is printed in GitHub-style markdown table.
    """
    def summarize_column(col_data):
        numeric_data = pd.to_numeric(col_data, errors='coerce')
        is_numeric = numeric_data.notna().sum() > 0
        return {
            "dtype": col_data.dtype,
            "count": col_data.notna().sum(),
            "pct_complete": round(col_data.notna().mean() * 100, 1),
            "n_unique": col_data.nunique(dropna=True)
        }

    summary_long = (
        pd.DataFrame([
            {"variable": col, **summarize_column(df[col])}
            for col in df.columns
        ])
        .sort_values(by="pct_complete", ascending=False)
        .reset_index(drop=True)
    )

    print(tabulate(summary_long, headers="keys", tablefmt="github", showindex=False))

Overwriting covariate columns onto dataset


In [18]:
#load in test
df_main = pd.read_csv("../../../data/finaldatasets/testdata/FixedRFdata (Copy).csv", na_values=["n/a", "missing", "-", "", "NA", "N/A"])


#load in covariate dataset
df_update = pd.read_csv("../../../data/finaldatasets/covariates/wheat_data_with_covariats.csv", na_values=["n/a", "missing", "-", "", "NA", "N/A"])


/tmp/ipykernel_200645/3241095904.py:2: DtypeWarning: Columns (6,7,13,14,15,16,17,20,22,23,29,31,32,34,35,36,37,38,39,46,47,71,72,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv("../../../data/finaldatasets/testdata/FixedRFdata (Copy).csv", na_values=["n/a", "missing", "-", "", "NA", "N/A"])


In [19]:
print(df_main.columns.tolist())
print(df_update.columns.tolist())

['id', 'Data.ID', 'Location', 'State.Region.County.Province', 'Country', 'Continent', 'Latitude..N.S.', 'Longitude..E.W.', 'Conversion.for.latitude', 'Conversion.for.longitude', 'Location.source', 'Observation.period', 'Wheat.Type', 'Crop.variety', 'Tillage.type', 'Planting.date', 'Treatment', 'Treatment.type', 'Grain.yield..tons.ha.1.', 'Mean.annual.temperature..Â.C.', 'Water.regime', 'Mean.annual.precipitation..mm.', 'Irrigation..mm.', 'Soil.type', 'Sand', 'Silt', 'Clay', 'Soil.organic.carbon..g.C.kg.1.', 'Soil.pH', 'N.type', 'N.rate..kg.N.ha.1.', 'N.fertilizer.management', 'P.type', 'P.rate..kg.P.ha.1.', 'Straw.return', 'Plastic.film.mulching', 'Emissions..yes.no.', 'Pest.prescence....64', 'Pest.detected...65', 'Pest.severity.score.......66', 'rainfed_start_date', 'rainfed_end_date', 'irr_start_date', 'irr_end_date', 'start_date', 'end_date', 'Planting.date.1', 'Harvesting.date.1', 'Soil_N', 'pr_irrigated', 'AEZ', 'Elevation', 'temp1', 'temp2', 'temp3', 'temp4', 'temp5', 'temp6', 't

In [20]:
# Merge on 'id', keeping all rows from df_main
merged = df_main.merge(df_update, on="id", how="left", suffixes=("", "_new"))

# For each overlapping column, overwrite old with new (where new is not null)
for col in df_update.columns:
    if col != "id" and col in df_main.columns:
        merged[col] = merged[f"{col}_new"].combine_first(merged[col])
        merged.drop(columns=[f"{col}_new"], inplace=True)

In [21]:

#sava dataset
df_main.to_csv("../../../data/finaldatasets/testdata/Overwrite.csv", index=False)

In [2]:
import cdsapi

dataset = "sis-biodiversity-era5-global"
request = {
    "variable": [
        "dry_spells",
        "growing_degree_days_during_growing_season_length"
    ],
    "derived_variable": ["number_of_occurrences"],
    "temporal_aggregation": ["annual"],
    "version": ["1_0"]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


2025-07-08 16:57:59,104 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-07-08 16:58:00,067 INFO [2025-01-29T00:00:00] This dataset is no longer supported by the data providers. Data and documentation are provided as is. Users are encouraged to use our [Forum](https://forum.ecmwf.int/) to raise any item of discussion with respect to this dataset.
2025-07-08 16:58:00,068 INFO Request ID is f088fc04-29e3-467e-990b-95d27b11daaa
2025-07-08 16:58:00,224 INFO status has been updated to accepted
2025-07-08 16:58:16,391 INFO status has been updated to running
2025-07-08 16:58:24,906 INFO status has been updated to successful


e4070a944ded3c3367f5ed6416fc48fd.zip:   0%|          | 0.00/76.5M [00:00<?, ?B/s]

'e4070a944ded3c3367f5ed6416fc48fd.zip'

Joining GDP Data: https://zenodo.org/records/13943886

In [11]:
#load in treedataset 
df = pd.read_csv("../../../../data/finaldatasets/testdata/RFgdpadded22-7.csv", na_values=["n/a", "missing", "-", "", "NA", "N/A"])

points = list(zip(df["Conversion.for.longitude"], df["Conversion.for.latitude"]))

#load in GDP raster data
tif_path = "../../../../data/finaldatasets/covariates/Covariates/rast_adm2_gdp_perCapita_1990_2022.tif"

/tmp/ipykernel_30476/1685558610.py:2: DtypeWarning: Columns (6,7,13,14,15,16,17,20,22,23,29,31,33,34,35,36,37,44,45,70,74,75,76) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../../../data/finaldatasets/testdata/RFgdpadded22-7.csv", na_values=["n/a", "missing", "-", "", "NA", "N/A"])


In [12]:
with rasterio.open(tif_path) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Width (pixels):", src.width)
    print("Height (pixels):", src.height)
    print("Number of bands:", src.count)
    print("Resolution (pixel size):", src.res)
    print("NoData value:", src.nodata)
    print("Data type:", src.dtypes[0])
    print("Transform:", src.transform)  


CRS: EPSG:4326
Bounds: BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
Width (pixels): 4320
Height (pixels): 2160
Number of bands: 33
Resolution (pixel size): (0.08333333333333333, 0.08333333333333333)
NoData value: 4294967295.0
Data type: uint32
Transform: | 0.08, 0.00,-180.00|
| 0.00,-0.08, 90.00|
| 0.00, 0.00, 1.00|


In [13]:
# Get unique years in your dataset
unique_years = sorted(df['year'].unique())
print(f"Years in dataset: {unique_years}")
print(f"Year range: {min(unique_years)} to {max(unique_years)}")

# Assume bands represent years 2005-2020
start_year = 2005

# Prepare output column if not present
if 'gdp_per_capita' not in df.columns:
    df['gdp_per_capita'] = np.nan

with rasterio.open(tif_path) as src:
    nodata = src.nodata
    band_count = src.count
    for year in unique_years:
        band_number = int(year - start_year + 1)
        if band_number < 1 or band_number > band_count:
            print(f"  Year {year}: Band {band_number} out of range (1-{band_count}), skipping.")
            continue

        print(f"Processing year {year} (band {band_number})...")

        year_mask = df['year'] == year
        year_data = df[year_mask]

        if len(year_data) > 0:
            year_points = [(row["Conversion.for.longitude"], row["Conversion.for.latitude"]) 
                           for _, row in year_data.iterrows()]
            try:
                gdp_values = list(src.sample(year_points, indexes=band_number))
                # Handle nodata and negative/zero values
                gdp_values = [
                    val[0] if (len(val) > 0 and val[0] != nodata and val[0] > 0) else np.nan
                    for val in gdp_values
                ]
                df.loc[year_mask, 'gdp_per_capita'] = gdp_values

                valid_gdp = [v for v in gdp_values if not np.isnan(v)]
                if valid_gdp:
                    print(f"  Year {year}: {len(valid_gdp)}/{len(gdp_values)} valid GDP values")
                    print(f"  GDP range: ${min(valid_gdp):,.0f} - ${max(valid_gdp):,.0f}")
                else:
                    print(f"  Year {year}: No valid GDP values found")
            except Exception as e:
                print(f"  Error processing year {year}: {e}")
        else:
            print(f"  No data found for year {year}")

# Check results
print(f"\nFinal Results:")
print(f"GDP values extracted: {df['gdp_per_capita'].notna().sum()}/{len(df)}")

# GDP statistics by year
print(f"\nGDP statistics by year:")
gdp_by_year = df.groupby('year')['gdp_per_capita'].agg(['count', 'mean', 'min', 'max']).round(0)
print(gdp_by_year)

# Overall GDP statistics
print(f"\nOverall GDP statistics:")
print(df['gdp_per_capita'].describe())

# Show sample with GDP data
print(f"\nSample results:")
sample_cols = ['year', 'Conversion.for.latitude', 'Conversion.for.longitude', 'gdp_per_capita']
print(df[sample_cols].head(10))

Years in dataset: [2005, 2006, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
Year range: 2005 to 2020
Processing year 2005 (band 1)...
  Year 2005: 788/788 valid GDP values
  GDP range: $2,314 - $49,348
Processing year 2006 (band 2)...
  Year 2006: 339/339 valid GDP values
  GDP range: $985 - $48,044
Processing year 2008 (band 4)...
  Year 2008: 433/433 valid GDP values
  GDP range: $1,159 - $91,389
Processing year 2009 (band 5)...
  Year 2009: 471/471 valid GDP values
  GDP range: $1,420 - $73,531
Processing year 2010 (band 6)...
  Year 2010: 589/589 valid GDP values
  GDP range: $1,368 - $55,262
Processing year 2011 (band 7)...
  Year 2011: 655/655 valid GDP values
  GDP range: $1,493 - $94,869
Processing year 2012 (band 8)...
  Year 2012: 598/598 valid GDP values
  GDP range: $1,563 - $70,624
Processing year 2013 (band 9)...
  Year 2013: 5152/5152 valid GDP values
  GDP range: $1,385 - $81,101
Processing year 2014 (band 10)...
  Year 2014: 5804/5804 v

In [14]:
summarize_dataframe(df)

| variable                                              | dtype   |   count |   pct_complete |   n_unique |
|-------------------------------------------------------|---------|---------|----------------|------------|
| id                                                    | int64   |   69430 |          100   |      69036 |
| temp8                                                 | float64 |   69430 |          100   |        282 |
| pr_irrigated                                          | float64 |   69428 |          100   |        191 |
| Data.ID                                               | object  |   69430 |          100   |         51 |
| Elevation                                             | int64   |   69430 |          100   |        196 |
| temp1                                                 | float64 |   69430 |          100   |        268 |
| temp2                                                 | float64 |   69430 |          100   |        270 |
| temp3                     

In [15]:
#Save the updated DataFrame
df.to_csv("../../../../data/finaldatasets/testdata/RFgdpadded24-7.csv", index=False)